# BLAM-ALL Training & TreeSHAP Analysis

This notebook trains BLAM-ALL model variants and performs TreeSHAP feature importance analysis.

**Requirements:**
- GPU Runtime (A100/T4/V100 recommended)
- Google Drive access for data and model storage

**Models Trained:**
- `BLAM-ALL` - All sky conditions
- `BLAM-ALL-CLEAR` - Clear sky only (BCM=0)
- `BLAM-ALL-CLOUDY` - Cloudy sky only (BCM=1)

---
## Cell 1: Check GPU & Mount Drive

In [ ]:
# Verify GPU availability
!nvidia-smi

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

---
## Cell 2: Clone Repository

In [ ]:
import os

# Clone your repository (replace with your actual repo URL)
REPO_URL = "https://github.com/karensylee/lst.git"  # <-- UPDATE THIS
REPO_DIR = "/content/lst"

if os.path.exists(REPO_DIR):
    print(f"Repository already exists at {REPO_DIR}")
    %cd {REPO_DIR}
    !git pull
else:
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

!ls -la

---
## Cell 3: Install Dependencies

In [ ]:
# Install required packages
!pip install -q polars xgboost optuna joblib scikit-learn

# Verify installations
import polars as pl
import xgboost as xgb
print(f"Polars: {pl.__version__}")
print(f"XGBoost: {xgb.__version__}")

---
## Cell 4: Configure Paths

Update these paths to match your Google Drive structure.

In [ ]:
import os

# === UPDATE THESE PATHS ===
# Path to your ML-ready dataset on Google Drive
DATA_PATH = "/content/drive/MyDrive/lst/datasets/processed/ML_READY_mesonet_goes_embeddings_2024.csv"

# Path where models will be saved on Google Drive
OUTPUT_DIR = "/content/drive/MyDrive/lst/models"

# === Set Environment Variables ===
os.environ['LST_DATA_PATH'] = DATA_PATH
os.environ['LST_PROJECT_ROOT'] = '/content/lst'

# Verify paths exist
print(f"Data exists: {os.path.exists(DATA_PATH)}")
print(f"Output dir: {OUTPUT_DIR}")

# Create output directories
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'xgb'), exist_ok=True)

---
## Cell 5: Patch settings.py for Colab Paths

Override the default paths in `config/settings.py` to use Google Drive.

In [ ]:
# Patch the settings module to use our Colab paths
import sys
sys.path.insert(0, '/content/lst')

from config import settings

# Override paths
settings.DATA_PATH = DATA_PATH
settings.OUTPUT_DIR = OUTPUT_DIR
settings.FIGURES_DIR = os.path.join(OUTPUT_DIR, 'figures')

print("Settings patched:")
print(f"  DATA_PATH: {settings.DATA_PATH}")
print(f"  OUTPUT_DIR: {settings.OUTPUT_DIR}")
print(f"  FEATURE_SETS available: {list(settings.FEATURE_SETS.keys())}")

---
## Cell 6: Train BLAM-ALL (All Sky Conditions)

Trains a single model on ALL data (no LOSO CV).

In [ ]:
%cd /content/lst
!python main.py --model_type BLAM-ALL

---
## Cell 7: Train BLAM-ALL-CLEAR (Clear Sky Only)

In [ ]:
!python main.py --model_type BLAM-ALL-CLEAR

---
## Cell 8: Train BLAM-ALL-CLOUDY (Cloudy Sky Only)

In [ ]:
!python main.py --model_type BLAM-ALL-CLOUDY

---
## Cell 9: Verify Trained Models

In [ ]:
import os

model_base = os.path.join(OUTPUT_DIR, 'xgb')

for variant in ['BLAM-ALL', 'BLAM-ALL-CLEAR', 'BLAM-ALL-CLOUDY']:
    model_path = os.path.join(model_base, variant, f"{variant}_model.joblib")
    exists = os.path.exists(model_path)
    status = "✅" if exists else "❌"
    print(f"{status} {variant}: {model_path}")

---
## Cell 10: Run TreeSHAP Analysis & Beeswarm Plots

This uses XGBoost's native TreeSHAP (`booster.predict(..., pred_contribs=True)`) on 10,000 samples per model.

In [ ]:
# Patch settings again for the analysis script
import sys
sys.path.insert(0, '/content/lst')

from config import settings
settings.DATA_PATH = DATA_PATH
settings.OUTPUT_DIR = OUTPUT_DIR

# Run the analysis
!python analysis/blam_shap_beeswarm.py

---
## Cell 11: Display Beeswarm Plots

In [ ]:
import os
from IPython.display import Image, display

shap_dir = os.path.join(OUTPUT_DIR, 'shap_analysis_blam_all')

for variant in ['BLAM-ALL', 'BLAM-ALL-CLEAR', 'BLAM-ALL-CLOUDY']:
    plot_path = os.path.join(shap_dir, f"shap_beeswarm_{variant}.jpg")
    if os.path.exists(plot_path):
        print(f"\n{'='*50}")
        print(f"{variant}")
        print('='*50)
        display(Image(filename=plot_path, width=800))
    else:
        print(f"❌ Plot not found: {plot_path}")

---
## Cell 12: View Feature Importance Rankings

In [ ]:
import pandas as pd
import os

shap_dir = os.path.join(OUTPUT_DIR, 'shap_analysis_blam_all')

for variant in ['BLAM-ALL', 'BLAM-ALL-CLEAR', 'BLAM-ALL-CLOUDY']:
    csv_path = os.path.join(shap_dir, f"shap_importance_{variant}.csv")
    if os.path.exists(csv_path):
        print(f"\n{'='*50}")
        print(f"{variant} - Top 15 Features")
        print('='*50)
        df = pd.read_csv(csv_path)
        display(df.head(15))
    else:
        print(f"❌ CSV not found: {csv_path}")

---
## Cell 13: Compare Feature Importance Across Variants

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

shap_dir = os.path.join(OUTPUT_DIR, 'shap_analysis_blam_all')

# Load all importance CSVs
dfs = {}
for variant in ['BLAM-ALL', 'BLAM-ALL-CLEAR', 'BLAM-ALL-CLOUDY']:
    csv_path = os.path.join(shap_dir, f"shap_importance_{variant}.csv")
    if os.path.exists(csv_path):
        dfs[variant] = pd.read_csv(csv_path).set_index('Feature')

if len(dfs) == 3:
    # Merge into comparison DataFrame
    comparison = pd.DataFrame({
        'All Sky': dfs['BLAM-ALL']['Mean_Abs_SHAP'],
        'Clear': dfs['BLAM-ALL-CLEAR']['Mean_Abs_SHAP'],
        'Cloudy': dfs['BLAM-ALL-CLOUDY']['Mean_Abs_SHAP']
    })
    
    # Sort by All Sky importance
    comparison = comparison.sort_values('All Sky', ascending=False)
    
    # Plot top 20
    fig, ax = plt.subplots(figsize=(12, 10))
    comparison.head(20).plot(kind='barh', ax=ax, width=0.8)
    ax.set_xlabel('Mean |SHAP| Value')
    ax.set_title('Feature Importance Comparison: All Sky vs Clear vs Cloudy')
    ax.invert_yaxis()
    plt.legend(title='Model Variant')
    plt.tight_layout()
    
    # Save comparison plot
    comparison_path = os.path.join(shap_dir, 'shap_comparison_variants.jpg')
    plt.savefig(comparison_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"\n✅ Saved comparison plot: {comparison_path}")
else:
    print("Not all variant CSVs found. Train all models first.")

---
## Cell 14: Backup Models to Drive (Optional)

In [ ]:
# Models are already saved to Google Drive via OUTPUT_DIR
# This cell just confirms the files are there

import os

print("Files saved to Google Drive:")
print("="*50)

for root, dirs, files in os.walk(os.path.join(OUTPUT_DIR, 'xgb')):
    for f in files:
        fpath = os.path.join(root, f)
        size_mb = os.path.getsize(fpath) / 1e6
        print(f"  {f}: {size_mb:.2f} MB")

print("\nSHAP Analysis:")
shap_dir = os.path.join(OUTPUT_DIR, 'shap_analysis_blam_all')
if os.path.exists(shap_dir):
    for f in os.listdir(shap_dir):
        print(f"  {f}")